# TP1 — Data Preparation & Random Forest (Breast Cancer with Noise)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/DFGSM2-IA-S2/blob/main/session2/tp1.ipynb)

## Objective

In session 1, we worked with a **clean dataset**. Real-world medical data is rarely clean — it contains **missing values**, **outliers**, and **categorical variables** that need preprocessing.

In this TP, we will:
1. Work with a **noised version** of the Breast Cancer dataset (missing values, outliers, categorical features)
2. Learn the full **data preparation pipeline**: imputation → outlier handling → encoding → scaling
3. Train a **Random Forest** classifier and compare it with logistic regression

### Methodology Recap (from Session 1)

The pipeline remains: **Explore → Visualize → Prepare → Train → Evaluate → Cross-validate**. This session focuses on the **Prepare** step, which we skipped in session 1 because the data was already clean.

> **Link to Session 1:** We reuse the same dataset and evaluation methodology. The new concepts are data preparation and a new model (Random Forest).

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, f1_score)

---
## 2. Load & Discover the Data

In session 1, the Breast Cancer Wisconsin dataset was **perfectly clean** — no missing values, no outliers, all numeric features. That's rare in real life!

This time, you receive a version of the same dataset that is closer to **real clinical data**:
- Some values are **missing** (equipment errors, incomplete patient records)
- Some measurements contain **outliers** (data entry mistakes, extreme cases)
- New **categorical columns** have been added (patient demographics)

Your first job as a data scientist: **discover and understand these issues** before any modeling.

### Exercise 1.1 — Load the dataset

Load the breast cancer dataset from the CSV file. Identify the column types and define useful variables.

**Why:** Always start by looking at your data. Unlike session 1 where the data was clean and ready to use, real datasets need inspection.

*Hint:*
```python
df = pd.read_csv('https://raw.githubusercontent.com/racousin/DFGSM2-IA-S2/refs/heads/main/session2/breast_cancer_noisy.csv')
target_names = np.array(['malignant', 'benign'])
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop('target').tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
```

In [ ]:
# YOUR CODE HERE


### Exercise 1.2 — Inspect data quality

Use `.info()` to see data types and missing values, and `.describe()` to check value ranges.

**Why:** Identify the three types of problems: missing values (NaN), outliers (extreme values), and categorical features (non-numeric).

*Hint:* `df.info()`, `df.describe()`

**Question:** How many missing values are there? Which features have suspicious values? Which columns are categorical?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

---
## 3. Visualize Data Issues

Before fixing problems, we need to **see** them. Visualization makes data issues obvious.

### Exercise 2.1 — Missing values heatmap

Visualize where missing values are located in the dataset.

**Why:** See the pattern of missing data — is it random or concentrated in certain features?

*Hint:* `sns.heatmap(df[numeric_cols].isnull(), cbar=True, yticklabels=False)` or `df[numeric_cols].isnull().sum().plot(kind='bar')`

In [ ]:
# YOUR CODE HERE


### Exercise 2.2 — Boxplots to detect outliers

Create boxplots for `mean radius` and `mean area` to visualize outliers.

**Why:** Boxplots show the median, quartiles, and **outliers** (points beyond 1.5× IQR). Extreme outliers will be clearly visible.

*Hint:* `fig, axes = plt.subplots(1, 2, figsize=(12, 5))`, `sns.boxplot(y=df['mean radius'], ax=axes[0])`

In [ ]:
# YOUR CODE HERE


### Exercise 2.3 — Categorical distributions

Plot bar charts for the categorical columns (`age_group`, `menopause_status`, `tumor_location`).

**Why:** Understand the distribution of categorical features and check if any category is very rare.

*Hint:* `df['age_group'].value_counts().plot(kind='bar')`

In [ ]:
# YOUR CODE HERE


---
## 4. Handle Missing Values

Missing values (NaN) must be handled before training — most ML models cannot work with NaN.

### Common strategies:
| Strategy | When to use |
|----------|------------|
| **Drop rows** | Very few missing values, large dataset |
| **Mean/Median imputation** | Numeric features, random missingness |
| **Most frequent imputation** | Categorical features |
| **Model-based imputation** | Complex patterns (advanced) |

We'll use `SimpleImputer` from sklearn:
- **Median** for numeric features (robust to outliers)
- **Most frequent** for categorical features

### Exercise 3.1 — Check missing values

Count missing values per column. How many total NaN values are there?

*Hint:* `df.isnull().sum()`, `df.isnull().sum().sum()`

In [ ]:
# YOUR CODE HERE


### Exercise 3.2 — Impute missing values

Use `SimpleImputer` with `strategy='median'` for numeric features and `strategy='most_frequent'` for categorical features.

**Why:** Median is more robust than mean when outliers are present (which is our case!).

*Hint:*
```python
num_imputer = SimpleImputer(strategy='median')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])
```

Verify with `df.isnull().sum().sum()` — should be 0.

In [ ]:
# YOUR CODE HERE


---
## 5. Handle Outliers

Outliers are extreme values that can distort model training. We'll use the **IQR method** to detect and clip them.

### IQR Method
- **IQR** = Q3 − Q1 (interquartile range)
- **Lower bound** = Q1 − 1.5 × IQR
- **Upper bound** = Q3 + 1.5 × IQR
- Values outside these bounds are **clipped** (set to the bound value)

**Why clipping instead of removing?** In medical data, we don't want to lose samples — every patient matters. Clipping preserves the sample while reducing the outlier's influence.

### Exercise 4.1 — Detect and clip outliers

For each numeric feature, clip values outside the IQR bounds. Show before/after statistics for `mean radius`.

*Hint:*
```python
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
```

In [ ]:
# YOUR CODE HERE


### Exercise 4.2 — Visualize before/after outlier handling

Create boxplots of `mean radius` and `mean area` after clipping. Compare with exercise 2.2.

**Why:** Verify that outliers have been effectively handled.

*Hint:* Same boxplot code as exercise 2.2.

In [ ]:
# YOUR CODE HERE


---
## 6. Encode Categorical Features

ML models work with **numbers**, not text. We need to convert categorical columns to numeric.

### One-Hot Encoding

For categorical features without natural order, we create **one binary column per category**:

| tumor_location | → | upper_inner | central | lower_inner |
|-----------|---|-----------------|------------------|---------------|
| upper_inner|   | 1               | 0                | 0             |
| central    |   | 0               | 1                | 0             |
| lower_inner|   | 0               | 0                | 1             |

**Why not assign numbers (upper_inner=0, central=1, lower_inner=2)?** Because the model would interpret this as an **order** (upper_inner < central < lower_inner), which may not be meaningful for all variables.

### Exercise 5.1 — One-hot encode categorical features

Use `pd.get_dummies()` to encode the categorical columns.

*Hint:* `df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)` — `drop_first=True` avoids redundancy.
you can also use sklearn 
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

**Note:** `drop_first=True` avoids the **dummy variable trap** (redundant columns).

**Question:** How many features do we have now after encoding?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

---
## 7. Feature Scaling

Same as session 1 — scale features so they have mean=0, std=1. This is important for logistic regression (gradient descent) and helps many other models too.

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

**Remember:** Fit on train, transform both train and test!

### Exercise 6.1 — Split and scale the data

Separate features (X) and target (y), split into train/test (80/20, `random_state=42`), then scale with `StandardScaler`.

*Hint:* Same pattern as session 1.

In [ ]:
# YOUR CODE HERE


---
## 8. Baseline: Logistic Regression

Before trying a new model, always establish a **baseline** with a model you already know. This gives you a reference point to judge whether the new model is actually better.

We'll reuse logistic regression from session 1 as our baseline.

### Exercise 7.1 — Train logistic regression baseline

Train a `LogisticRegression(max_iter=5000)` on the scaled data. Report accuracy and classification report.

**Why:** This establishes the performance we need to beat with Random Forest.

*Hint:* Same as session 1 exercises.

In [ ]:
# YOUR CODE HERE


---
## 9. Random Forest

### What is a Random Forest?

A **Random Forest** is an **ensemble** of many **decision trees**, each trained on a random subset of the data and features. The final prediction is the **majority vote** of all trees.

```
        Data
       / | \
   Tree1 Tree2 Tree3 ... TreeN
     |     |     |         |
   pred1 pred2 pred3 ... predN
          \   |   /
        Majority Vote
          → Final prediction
```

### Why Random Forest?

| Property | Logistic Regression | Random Forest |
|----------|-------------------|---------------|
| **Decision boundary** | Linear (straight line) | Non-linear (complex shapes) |
| **Feature scaling needed** | Yes | No |
| **Handles outliers** | Sensitive | Robust |
| **Interpretability** | Coefficients | Feature importances |
| **Risk of overfitting** | Low | Medium (controllable) |

### Key Idea: Ensemble Learning

Instead of relying on one model, combine many "weak" models into a strong one. Each tree sees a different random subset of the data, so they make **different errors**. When combined, the errors cancel out.

> **Medical analogy:** It's like getting opinions from multiple doctors — each may have a different perspective, but the consensus is usually more reliable than any single opinion.

### Exercise 8.1 — Train a Random Forest

Train a `RandomForestClassifier(n_estimators=100, random_state=42)` and compare its accuracy with the logistic regression baseline.

**Note:** Random Forest doesn't require scaled features, but we use the same scaled data for fair comparison.

*Hint:* Same `.fit()` / `.predict()` pattern.

In [ ]:
# YOUR CODE HERE


### Exercise 8.2 — Compare confusion matrices

Display confusion matrices side by side for logistic regression and random forest.

**Why:** See if the models make different types of errors.

*Hint:* `fig, axes = plt.subplots(1, 2, figsize=(12, 5))`

In [ ]:
# YOUR CODE HERE


### Exercise 8.3 — Feature importances

Plot the top 15 most important features according to the Random Forest.

**Why:** Random Forest can tell us which features matter most — this is valuable for understanding what drives the diagnosis.

*Hint:*
```python
importances = rf_model.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[-15:]
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), feature_names[indices])
```

**Question:** Which features are most important? Do they match the correlations from session 1?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

---
## 10. Hyperparameter Tuning

### What are Hyperparameters?

Hyperparameters are settings we choose **before** training — the model doesn't learn them from data.

| Hyperparameter | What it controls | Too low | Too high |
|---------------|-----------------|---------|----------|
| `n_estimators` | Number of trees | Unstable predictions | Slow, diminishing returns |
| `max_depth` | Maximum depth of each tree | Underfitting (too simple) | Overfitting (memorizes noise) |
| `min_samples_split` | Minimum samples to split a node | Overfitting | Underfitting |



### Exercise 9.0 — Test hyperparamters
Train a `RandomForestClassifier(n_estimators=try, max_depth=try, min_samples_split=try, random_state=42)` and compare its accuracies.

In [ ]:
# YOUR CODE HERE


### Exercise 9.1 — Grid search for best Random Forest

### GridSearchCV

`GridSearchCV` tries all combinations of hyperparameters and picks the best one using cross-validation:

```
n_estimators: [50, 100, 200]  ×  max_depth: [3, 5, 10, None]  ×  min_samples_split: [2, 5]
= 3 × 4 × 2 = 24 combinations, each evaluated with 5-fold CV = 120 model fits
```

Use `GridSearchCV` to find the best hyperparameters. Report the best parameters and best score.

*Hint:*
```python
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")
```

In [ ]:
# YOUR CODE HERE


### Exercise 9.2 — Evaluate best model on test set

Use the best model from grid search to predict on the test set. Show accuracy and classification report.

*Hint:* `grid_search.best_estimator_` gives the best model, already fitted.

**Question:** Did tuning improve performance compared to the default Random Forest?

In [ ]:
# YOUR CODE HERE


*YOUR ANSWER HERE*

---
## 11. Cross-Validation & Final Comparison

### Exercise 10.1 — Cross-validation comparison

Run 5-fold cross-validation for both logistic regression and the best random forest. Create a comparison table.

*Hint:*
```python
lr_scores = cross_val_score(LogisticRegression(max_iter=5000), X_scaled, y, cv=5, scoring='accuracy')
rf_scores = cross_val_score(grid_search.best_estimator_, X_scaled, y, cv=5, scoring='accuracy')
```

Display results in a summary table.

In [ ]:
# YOUR CODE HERE


---
## 12. Summary

### What We Learned

| Step | Technique | Why |
|------|-----------|-----|
| **Missing values** | `SimpleImputer(median)` | Fill NaN values without losing samples |
| **Outliers** | IQR clipping | Reduce extreme values without removing samples |
| **Categorical encoding** | One-hot encoding (`get_dummies`) | Convert text categories to numeric features |
| **Feature scaling** | `StandardScaler` | Normalize feature ranges |
| **Baseline** | Logistic Regression | Reference performance to beat |
| **New model** | Random Forest | Ensemble of decision trees — often more powerful |
| **Tuning** | `GridSearchCV` | Find optimal hyperparameters via cross-validation |

### Data Preparation Pipeline

```
Raw Data → Handle Missing → Handle Outliers → Encode Categories → Scale → Train/Test Split → Model
```

**Key insight:** Data preparation is often more important than model choice. A well-prepared dataset with a simple model often beats a complex model on messy data.